# Use Case 4: QML Time Series Forecasting (Electricity Prices)

This use case demonstrates electricity price time-series forecasting using a **Quantum Neural Network (QNN) Regressor**.

- **Author:** Jiří Tomčala (IT4Innovations National Supercomputing Center)
- **Quantum Architecture:** 8-qubit Parameterized Quantum Circuit (`ZFeatureMap` + `EfficientSU2` ansatz with 80 trainable parameters $\theta$).
- **Data:** Real historical hourly electricity market prices from OTE-CR (Czech Market Operator).
- **Environment:** Runs directly under LUMI-IQ module (`module load lumi-iq`).

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings("ignore", message=".*IProgress not found.*")

import numpy as np
import matplotlib.pyplot as plt
from qiskit_machine_learning.algorithms.regressors import NeuralNetworkRegressor

print("QML Time Series environment initialized.")

# Backend selection: "SIM" (fast local simulation) or "VLQ" (real QPU via LEXIS)
TARGET_BACKEND = "SIM"  # Change to "VLQ" to dispatch 1-hour ahead forecast circuit to VLQ


### 1. Load Real Historical Market Data
Load and normalize hourly electricity price series into 8-hour lookback feature windows.

In [ ]:
data_orig = np.load("data.npy", allow_pickle=True)[96:]
n = 360

min_val = min(data_orig[0:(8+n)])
max_val = max(data_orig[0:(8+n)])
data = ((2 * np.pi) * (data_orig - min_val) / (max_val - min_val)) - np.pi

inputs = []
for i in range(n):
    inputs.append(data[i:(i+8)])
inputs = np.array(inputs)

outputs = np.array(data[8:(8+n)])
outputs_norm = ((outputs - min(outputs)) / (max(outputs) - min(outputs))) - 0.5
outputs_orig = np.array(data_orig[8:(8+n)])
outputs_train_orig = outputs_orig[0:240]

print(f"Loaded {len(data_orig)} time steps. Testing window: {n} hours ahead.")


### 2. Load Pre-trained QNN Model & Perform Inference
Evaluate the 8-qubit QNN model on the test sequence.

In [ ]:
modelfile = "QNN_0.7706084854606758-model_state.dill"
reg = NeuralNetworkRegressor.from_dill(modelfile)

outputs_predicted_norm = reg.predict(inputs)[:, 0]
outputs_predicted_norm_pi = (outputs_predicted_norm + 0.5) * (max(outputs) - min(outputs)) + min(outputs)
outputs_predicted_orig = (outputs_predicted_norm_pi + np.pi)/(2*np.pi) * (max_val - min_val) + min_val

score = reg.score(inputs, outputs_norm)
print(f"Forecasting Accuracy Score: {score:.4f} ({score*100:.2f}%)")


### 3. Forecast Plot
Compare real historical prices against the QNN prediction.

In [ ]:
plt.figure(figsize=(16, 5))
plt.plot(range(len(outputs_orig)), outputs_orig, "r--", label="Real Market Prices (OTE-CR)")
plt.plot(range(len(outputs_train_orig)), outputs_train_orig, "b.", label="Training Window")
plt.plot(range(len(outputs_predicted_orig)), outputs_predicted_orig, "g-", linewidth=2, label="QNN Forecast (8-qubit PQC)")
plt.title(f"Electricity Price 1-Hour Ahead Forecasting (Average Precision: {score*100:.2f}%)", fontsize=13, fontweight="bold")
plt.xlabel("Time [Hours]", fontsize=11)
plt.ylabel("Price [EUR/MWh]", fontsize=11)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig("result_time_series.png", dpi=150)
plt.show()
